Open AI Embedding Models

LangChain enables us to access Open AI embedding models which include the newest models: a smaller and highly efficient text-embedding-3-small model, and a larger and more powerful text-embedding-3-large model.

In [8]:
from langchain_openai import OpenAIEmbeddings
openai_embed_model = OpenAIEmbeddings(model='text-embedding-3-small')

In [4]:
docs = [
 'Quantum mechanics describes the behavior of very small particles.',
 'Photosynthesis is the process by which green plants make food using sunlight.',
 'Artificial Intelligence aims to create machines that can think and learn.',
 'The pyramids of Egypt are historical monuments that have stood for thousands of years.',
 'New Delhi is the capital of India and the seat of all three branches of the Government of India.',
 'Biology is the study of living organisms and their interactions with the environment.',
 'Music therapy can aid in the mental well-being of individuals.',
 'Mumbai is the financial capital and the most populous city of India. It is the financial, commercial, and entertainment capital of South Asia.',
 'The Milky Way is just one of billions of galaxies in the universe.',
 'Economic theories help understand the distribution of resources in society.',
 'Kolkata is the de facto cultural capital of India and a historically and culturally significant city. Calcutta served as the de facto capital of India until 1911.',
 'Yoga is an ancient practice that involves physical postures and meditation.'
]

In [9]:
# collection_metadata is - How should Chroma store vectors and calculate distance?
# ction_metadata={"hnsw:space": "cosine"} means use hnsw index and use cosine to calculate similarity. default is l2 whihc is euclidean distance.
from langchain_chroma import Chroma

# create vector DB of docs and embeddings - takes 1 min on Colab
chroma_db = Chroma.from_texts(texts=docs, collection_name='db_docs',
                              # need to set the distance function to cosine else it uses euclidean by default
                              # check https://docs.trychroma.com/guides#changing-the-distance-function
                              collection_metadata={"hnsw:space": "cosine"},
                              embedding=openai_embed_model)

In [8]:
# Return top 3 documents which has highest similarity score. which metric to be used to calculate similarity score that we will define during
# retriever setup (in this case we have define co-sine)
similarity_retriever = chroma_db.as_retriever(search_type="similarity",
                                              search_kwargs={"k": 3})

In [9]:
# This wil retrieve top 3 chunks
query = "what is the capital of India?"
top3_docs = similarity_retriever.invoke(query)
top3_docs

[Document(id='a262cf78-e605-4559-924e-128f0090bc8b', metadata={}, page_content='New Delhi is the capital of India and the seat of all three branches of the Government of India.'),
 Document(id='ab5eed47-b400-4393-ae60-e153e0ccf18c', metadata={}, page_content='Mumbai is the financial capital and the most populous city of India. It is the financial, commercial, and entertainment capital of South Asia.'),
 Document(id='2b368df5-4609-4134-8d12-fa565c11042d', metadata={}, page_content='Kolkata is the de facto cultural capital of India and a historically and culturally significant city. Calcutta served as the de facto capital of India until 1911.')]

In [11]:
query = "what is the old capital of India?"
top3_docs = similarity_retriever.invoke(query)
top3_docs

[Document(id='2b368df5-4609-4134-8d12-fa565c11042d', metadata={}, page_content='Kolkata is the de facto cultural capital of India and a historically and culturally significant city. Calcutta served as the de facto capital of India until 1911.'),
 Document(id='a262cf78-e605-4559-924e-128f0090bc8b', metadata={}, page_content='New Delhi is the capital of India and the seat of all three branches of the Government of India.'),
 Document(id='ab5eed47-b400-4393-ae60-e153e0ccf18c', metadata={}, page_content='Mumbai is the financial capital and the most populous city of India. It is the financial, commercial, and entertainment capital of South Asia.')]

In [12]:
# This fetch top-10 based simlarity and apply Maximal Marginal Relevance (MMR) and return 3 chunks having highest mrr
mmr_retriever = chroma_db.as_retriever(search_type="mmr",
                                       search_kwargs={"k": 3,
                                                      'fetch_k': 10})

In [13]:
query = "what is the capital of India?"
top3_docs = mmr_retriever.invoke(query)
top3_docs

[Document(id='a262cf78-e605-4559-924e-128f0090bc8b', metadata={}, page_content='New Delhi is the capital of India and the seat of all three branches of the Government of India.'),
 Document(id='ab5eed47-b400-4393-ae60-e153e0ccf18c', metadata={}, page_content='Mumbai is the financial capital and the most populous city of India. It is the financial, commercial, and entertainment capital of South Asia.'),
 Document(id='2b368df5-4609-4134-8d12-fa565c11042d', metadata={}, page_content='Kolkata is the de facto cultural capital of India and a historically and culturally significant city. Calcutta served as the de facto capital of India until 1911.')]

In [14]:
query = "what is the old capital of India?"
top3_docs = mmr_retriever.invoke(query)
top3_docs

[Document(id='2b368df5-4609-4134-8d12-fa565c11042d', metadata={}, page_content='Kolkata is the de facto cultural capital of India and a historically and culturally significant city. Calcutta served as the de facto capital of India until 1911.'),
 Document(id='a262cf78-e605-4559-924e-128f0090bc8b', metadata={}, page_content='New Delhi is the capital of India and the seat of all three branches of the Government of India.'),
 Document(id='1af2963e-afc4-4768-a150-018f1b889069', metadata={}, page_content='Yoga is an ancient practice that involves physical postures and meditation.')]

Similarity with Threshold Retrieval

We use cosine similarity here and retrieve the top 3 similar documents based on the user input query and also introduce a cutoff to not return any documents which are below a certain similarity threshold. it will use cosine (we set that at DB level) and return top-3 chunk having cosine score more than 30%

In [15]:
similarity_threshold_retriever = chroma_db.as_retriever(search_type="similarity_score_threshold",
                                                        search_kwargs={"k": 3,
                                                                       "score_threshold": 0.3})

In [16]:
query = "what is the capital of India?"
top3_docs = similarity_threshold_retriever.invoke(query)
top3_docs

[Document(id='a262cf78-e605-4559-924e-128f0090bc8b', metadata={}, page_content='New Delhi is the capital of India and the seat of all three branches of the Government of India.'),
 Document(id='ab5eed47-b400-4393-ae60-e153e0ccf18c', metadata={}, page_content='Mumbai is the financial capital and the most populous city of India. It is the financial, commercial, and entertainment capital of South Asia.'),
 Document(id='2b368df5-4609-4134-8d12-fa565c11042d', metadata={}, page_content='Kolkata is the de facto cultural capital of India and a historically and culturally significant city. Calcutta served as the de facto capital of India until 1911.')]

Custom Retriever with Similarity Scores + Thresholding

Here we will create a custom retriever which will:

Retrieve documents with cosine distance
Convert to similarity score and apply thresholding
Return topk documents above a similarity threshold
we can extract the documents directly from the vector DB also without retriever using "similarity_search_with_score", we set cosine as distance algo at our database, this finction will calculate the cosine distnace and which ever documents top-3 has smaller angles are closer are retrived.

cosine similarity= 1- cosine distance

Thsi i why most relevent retrieved documents has smaller score.

but when we set retriever we set similarity or mmr which convert co-sine distance to cosine similarity.

In [17]:
query = 'how do plants make food?'
chroma_db.similarity_search_with_score(query, k=3)

[(Document(id='0619497f-392f-4206-9013-e82d55eecb2e', metadata={}, page_content='Photosynthesis is the process by which green plants make food using sunlight.'),
  0.35373449325561523),
 (Document(id='58bdda8b-2e39-44ff-a4f1-72730cdca1cb', metadata={}, page_content='Biology is the study of living organisms and their interactions with the environment.'),
  0.8317229151725769),
 (Document(id='1853250e-45be-4f4e-9d72-03fc9fbc9132', metadata={}, page_content='Artificial Intelligence aims to create machines that can think and learn.'),
  0.876487135887146)]

In [18]:
# converts cosine distance to similarity; cosine_similarity = 1 - cosine_distance
cosine_sim = chroma_db._select_relevance_score_fn()
cosine_sim(0.35375)

0.64625

In [19]:
# Lets create customer retriever as langchain chain
from typing import List
from langchain_core.documents import Document
from langchain_core.runnables import chain

In [20]:
@chain
def custom_retriever(query: str, topk=3, threshold_score=0.3) -> List[Document]:
    # get similarity conversion function (converts cosine distance to similarity)
    cosine_sim = chroma_db._select_relevance_score_fn()
    # get topk documents with lowest cosine distance
    docs, scores = zip(*chroma_db.similarity_search_with_score(query, k=topk))
    final_docs = []
    for doc, score in zip(docs, scores):
        # convert cosine distance to similarity
        score = cosine_sim(score)
        doc.metadata["score"] = round(score, 3)
        # check if score is above threshold
        if score > threshold_score:
            final_docs.append(doc)

    return final_docs

In [22]:
# Lets use custom retriever to retrieve documents
query = "what is the financial capital of India?"
top3_docs = custom_retriever.invoke(query, topk=3, threshold_score=0.51)
top3_docs

[Document(id='ab5eed47-b400-4393-ae60-e153e0ccf18c', metadata={'score': 0.69}, page_content='Mumbai is the financial capital and the most populous city of India. It is the financial, commercial, and entertainment capital of South Asia.'),
 Document(id='a262cf78-e605-4559-924e-128f0090bc8b', metadata={'score': 0.54}, page_content='New Delhi is the capital of India and the seat of all three branches of the Government of India.')]

Multi Query Retrieval

In [23]:
from langchain_openai import ChatOpenAI

chatgpt = ChatOpenAI(model_name="gpt-4o-mini", temperature=0)

In [29]:
from langchain_classic.retrievers import MultiQueryRetriever

In [32]:
# Set logging for the queries
import logging

similarity_retriever = chroma_db.as_retriever(search_type="similarity",
                                              search_kwargs={"k": 3})

# gpt-4o-mini will be used as LLM to generate multiple variants of original question and then each of the artificial question are send to retriever
# and it retrieve the most similar documents.
mq_retriever = MultiQueryRetriever.from_llm(
    retriever=similarity_retriever, llm=chatgpt
)

logging.basicConfig(level=logging.INFO)

logging.getLogger(
    "langchain_classic.retrievers.multi_query"
).setLevel(logging.INFO)

In [33]:
query = "financial capital of India?"
docs = mq_retriever.invoke(query)
docs

INFO:langchain_classic.retrievers.multi_query:Generated queries: ['What is the capital city of India in terms of its financial significance?  ', 'Which city serves as the financial hub of India?  ', 'Can you tell me about the financial capital of India?']


[Document(id='ab5eed47-b400-4393-ae60-e153e0ccf18c', metadata={}, page_content='Mumbai is the financial capital and the most populous city of India. It is the financial, commercial, and entertainment capital of South Asia.'),
 Document(id='a262cf78-e605-4559-924e-128f0090bc8b', metadata={}, page_content='New Delhi is the capital of India and the seat of all three branches of the Government of India.'),
 Document(id='2b368df5-4609-4134-8d12-fa565c11042d', metadata={}, page_content='Kolkata is the de facto cultural capital of India and a historically and culturally significant city. Calcutta served as the de facto capital of India until 1911.')]

Contextual Compression Retrieval

LLM-based re-ranker

In [36]:
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor

In [37]:
# simple cosine distance based retriever
similarity_retriever = chroma_db.as_retriever(search_type="similarity",
                                              search_kwargs={"k": 3})

# extracts from each document only the content that is relevant to the query
compressor = LLMChainExtractor.from_llm(llm=chatgpt)

# retrieves the documents similar to query and then applies the compressor
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=similarity_retriever
)

In [38]:
query = "what is the financial capital of India?"
docs = compression_retriever.invoke(query)
docs

[Document(metadata={}, page_content='Mumbai is the financial capital and the most populous city of India. It is the financial, commercial, and entertainment capital of South Asia.')]

In [39]:
query = "what is the old capital of India?"
docs = compression_retriever.invoke(query)
docs

[Document(metadata={}, page_content='Calcutta served as the de facto capital of India until 1911.')]

The LLMChainFilter is slightly simpler but more robust compressor that uses an LLM chain to decide which of the initially retrieved documents to filter out and which ones to return, without manipulating the document contents.

In [40]:
from langchain_classic.retrievers.document_compressors import LLMChainFilter

In [41]:
#  decides which of the initially retrieved documents to filter out and which ones to return
_filter = LLMChainFilter.from_llm(llm=chatgpt)

# retrieves the documents similar to query and then applies the filter
compression_retriever = ContextualCompressionRetriever(
    base_compressor=_filter, base_retriever=similarity_retriever
)

In [42]:
query = "what is the financial capital of India?"
docs = compression_retriever.invoke(query)
docs

[Document(id='ab5eed47-b400-4393-ae60-e153e0ccf18c', metadata={}, page_content='Mumbai is the financial capital and the most populous city of India. It is the financial, commercial, and entertainment capital of South Asia.')]

Ensemble Retrieval

The EnsembleRetriever takes a list of retrievers as input and ensemble the results of each of their retrievals and rerank the results based on the Reciprocal Rank Fusion algorithm.

In [44]:
from langchain_classic.retrievers import EnsembleRetriever

In [45]:
# simple cosine distance based retriever
similarity_retriever = chroma_db.as_retriever(search_type="similarity",
                                              search_kwargs={"k": 3})

# retrieves the documents similar to query and then applies the filter
compression_retriever = ContextualCompressionRetriever(
    base_compressor=_filter, base_retriever=similarity_retriever
)


ensemble_retriever = EnsembleRetriever(
    retrievers=[compression_retriever, similarity_retriever],
    weights=[0.7, 0.3]
)

In [46]:
query = "what is the financial capital of India?"
docs = ensemble_retriever.invoke(query)
docs

[Document(id='ab5eed47-b400-4393-ae60-e153e0ccf18c', metadata={}, page_content='Mumbai is the financial capital and the most populous city of India. It is the financial, commercial, and entertainment capital of South Asia.'),
 Document(id='a262cf78-e605-4559-924e-128f0090bc8b', metadata={}, page_content='New Delhi is the capital of India and the seat of all three branches of the Government of India.'),
 Document(id='2b368df5-4609-4134-8d12-fa565c11042d', metadata={}, page_content='Kolkata is the de facto cultural capital of India and a historically and culturally significant city. Calcutta served as the de facto capital of India until 1911.')]

Self Query Retrieval

User query is unstructured text. This query first goes to retriever; retriever first applies the query constructor which identifies:
	- Which part of the query is actual question.
	- Which parts are filter in the query which can help to filter documents based on metadata attributes.
Output of query constructor are sent to query translator which translate it to more appropriate query for vector database which include two things:
	- Text query which help to calculate embeddings similarity
	- Filters which helps on metadata filtering based on metadata filters and attribute. 


your documents look like:

In [34]:
from langchain_core.documents import Document

documents = [
    Document(
        page_content="Leave policy for HR employees in 2024",
        metadata={"department": "HR", "year": 2024, "type": "policy"}
    ),
    Document(
        page_content="IT security policy updated in 2023",
        metadata={"department": "IT", "year": 2023, "type": "policy"}
    ),
    Document(
        page_content="Finance audit guidelines 2024",
        metadata={"department": "Finance", "year": 2024, "type": "guideline"}
    )
]

In [35]:
# Create Vector Store
vectorstore1 = Chroma.from_documents(documents,embedding=openai_embed_model)

In [36]:
# Define Metadata Schema - This tells the LLM what filters are allowed.
from langchain_classic.chains.query_constructor.schema import AttributeInfo

metadata_field_info = [
    AttributeInfo(
        name="department",
        description="Department that owns the document",
        type="string"
    ),
    AttributeInfo(
        name="year",
        description="Year the document was published",
        type="integer"
    ),
    AttributeInfo(
        name="type",
        description="Type of document: policy, guideline, procedure",
        type="string"
    ),
]

# Create Self Query Retriever

In [37]:
from langchain_classic.retrievers import SelfQueryRetriever
from langchain_community.query_constructors.chroma import ChromaTranslator
from langchain_openai import ChatOpenAI
from lark import Lark

In [38]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [39]:
retriever = SelfQueryRetriever.from_llm(
    llm=llm,
    vectorstore=vectorstore1,
    document_contents="Company policies and guidelines",
    metadata_field_info=metadata_field_info,
    structured_query_translator=ChromaTranslator(),
    verbose=True,
)

In [42]:
query = "Show me IT security policies from 2023"

docs = retriever.invoke(query)

In [43]:
docs

[Document(id='7a7de11e-13c1-4aeb-9f60-c51f47b16182', metadata={'type': 'policy', 'department': 'IT', 'year': 2023}, page_content='IT security policy updated in 2023')]

What happes in backend is, LLM convert query as below <br>

{
  "query": "leave",
  "filter": {
    "department": "HR",
    "year": 2024,
    "type": "policy"
  }
}

and then LangChain run below
vectorstore.similarity_search(
    query="leave",
    filter={"department": "HR", "year": 2024, "type": "policy"}
)

##Hybrid Search Retrieval

#Hybrid Search = Semantic search (vectors) + Keyword search (BM25) combined

In [44]:
similarity_retriever = chroma_db.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

In [45]:
from langchain_community.retrievers import BM25Retriever

In [16]:
documents = [
    Document(page_content=text,metadata={})
    for text in docs
]

In [17]:
documents

[Document(metadata={}, page_content='Quantum mechanics describes the behavior of very small particles.'),
 Document(metadata={}, page_content='Photosynthesis is the process by which green plants make food using sunlight.'),
 Document(metadata={}, page_content='Artificial Intelligence aims to create machines that can think and learn.'),
 Document(metadata={}, page_content='The pyramids of Egypt are historical monuments that have stood for thousands of years.'),
 Document(metadata={}, page_content='New Delhi is the capital of India and the seat of all three branches of the Government of India.'),
 Document(metadata={}, page_content='Biology is the study of living organisms and their interactions with the environment.'),
 Document(metadata={}, page_content='Music therapy can aid in the mental well-being of individuals.'),
 Document(metadata={}, page_content='Mumbai is the financial capital and the most populous city of India. It is the financial, commercial, and entertainment capital of S

In [55]:
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(docs)
bm25_retriever.k = 3

In [56]:

from langchain_classic.retrievers import EnsembleRetriever
hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, similarity_retriever],
    weights=[0.5, 0.5]   # keyword weight, vector weight
)

In [57]:
query = "how do plants make food"
docs = hybrid_retriever.invoke(query)

In [58]:
docs

[Document(id='3646a477-406d-4e76-8a01-0d4dccd57838', metadata={}, page_content='Photosynthesis is the process by which green plants make food using sunlight.'),
 Document(id='82f8cd62-a541-4e26-bb3e-6898379317ce', metadata={}, page_content='Biology is the study of living organisms and their interactions with the environment.'),
 Document(metadata={'department': 'HR', 'year': 2024, 'type': 'policy'}, page_content='Leave policy for HR employees in 2024'),
 Document(id='e250c9f2-a2ff-40a0-a270-c68e267fdff0', metadata={}, page_content='Economic theories help understand the distribution of resources in society.')]

In [60]:
for d in docs:
    if(d.id):
        print(d.page_content)
    

Photosynthesis is the process by which green plants make food using sunlight.
Biology is the study of living organisms and their interactions with the environment.
Economic theories help understand the distribution of resources in society.


In [71]:
docs_with_id = [doc for doc in docs if getattr(doc, "id", None)]

for doc in docs_with_id:
    print(doc)
    print()  # blank line between documents


page_content='Photosynthesis is the process by which green plants make food using sunlight.'

page_content='Biology is the study of living organisms and their interactions with the environment.'

page_content='Economic theories help understand the distribution of resources in society.'

